# Step 10: Trying to improve the model

Three ideas, all compared on the **validation** set. The test set is not used here.

The current model (gradient boosting, 50 trees) scores ROC-AUC 0.680 and PR-AUC 0.239
on validation. Earlier bootstrapping showed these scores move by about ±0.01 by chance,
so an idea only counts as an improvement if it beats that clearly.

In [ ]:
import os

os.chdir("..")   # run from the project root so `src` can be imported


from src import baseline, clean, config, features, preprocess, split, train

In [2]:
def validation_scores(model, train_df, val_df, columns):
    X, y = train_df[columns], train_df[config.TARGET]
    X_val, y_val = val_df[columns], val_df[config.TARGET]
    p = model.fit(X, y).predict_proba(X_val)[:, 1]
    result = baseline.scores(y_val, p)
    return {"roc_auc": round(result["roc_auc"], 3), "pr_auc": round(result["pr_auc"], 3)}

train_df, val_df, _ = baseline.load_splits()
columns = preprocess.NUMERIC + preprocess.CATEGORICAL
print("current model:", validation_scores(train.make_boosting(50), train_df, val_df, columns))

current model: {'roc_auc': 0.68, 'pr_auc': 0.239}


## Idea 1: the patient's own history

Many patients have several visits. Two new columns: how many earlier visits this patient
has in the data, and how many of them ended in a readmission within 30 days.

**Only earlier visits may be used.** Information from later visits would not exist yet
at the time of the prediction. The data has no dates, so first we check that
`encounter_id` follows time.

In [3]:
raw = clean.load_raw()
by_patient = raw.sort_values(["patient_nbr", "encounter_id"])

# If encounter_id follows time, "inpatient stays in the previous year" should
# usually go up from one visit of a patient to the next.
change = by_patient.groupby("patient_nbr")["number_inpatient"].diff().dropna()
print(f"next visit has more previous stays:  {(change > 0).mean():.0%}")
print(f"next visit has fewer previous stays: {(change < 0).mean():.0%}")

next visit has more previous stays:  74%
next visit has fewer previous stays: 10%


Mostly up, so `encounter_id` is close to the order in time.

This also shows why later visits must never be used: a visit that ended **without** any
readmission almost never has a later visit in the data. A column like "has a later
visit" would give the answer away.

In [4]:
by_patient["has_later_visit"] = by_patient.groupby("patient_nbr")["encounter_id"].shift(-1).notna()
by_patient.groupby("readmitted")["has_later_visit"].mean().round(3)

readmitted
<30    0.716
>30    0.608
NO     0.009
Name: has_later_visit, dtype: float64

In [5]:
def add_history(raw):
    """Count each patient's earlier visits and earlier 30-day readmissions."""
    df = raw.sort_values("encounter_id").copy()
    readmitted_30 = (df["readmitted"] == "<30").astype(int)
    df["prior_visits"] = df.groupby("patient_nbr").cumcount()
    df["prior_readmissions_30"] = readmitted_30.groupby(df["patient_nbr"]).cumsum() - readmitted_30
    return df.sort_index()

data = features.add_diagnosis_groups(clean.clean(add_history(raw)))
data = preprocess.prepare(data)
h_train, h_val, _ = split.split_by_patient(data)
grouper = preprocess.SpecialtyGrouper().fit(h_train)
h_train, h_val = grouper.transform(h_train), grouper.transform(h_val)

history = ["prior_visits", "prior_readmissions_30"]


def boosting_with(numeric_columns, n_trees=50):
    """Same model as train.make_boosting, but with our own list of numeric columns."""
    from sklearn.compose import ColumnTransformer
    from sklearn.ensemble import HistGradientBoostingClassifier
    from sklearn.pipeline import make_pipeline
    from sklearn.preprocessing import OneHotEncoder

    columns_step = ColumnTransformer([
        ("numeric", "passthrough", numeric_columns),
        ("categorical", OneHotEncoder(handle_unknown="ignore", sparse_output=False),
         preprocess.CATEGORICAL),
    ])
    model = HistGradientBoostingClassifier(max_iter=n_trees, learning_rate=0.1,
                                           min_samples_leaf=20, early_stopping=False,
                                           random_state=0)
    return make_pipeline(columns_step, model)


numeric_with_history = preprocess.NUMERIC + history
print("with history:", validation_scores(boosting_with(numeric_with_history), h_train, h_val,
                                         numeric_with_history + preprocess.CATEGORICAL))

with history: {'roc_auc': 0.684, 'pr_auc': 0.239}


Almost no change. `number_inpatient` already tells the model how often the patient
has been in hospital, so the new columns add little.

## Idea 2: tuning gradient boosting

A smaller learning rate with more trees, smaller trees, and stronger regularisation.

In [6]:
settings = [
    # learning rate, trees, max leaves per tree, L2 regularisation
    (0.05, 100, 15, 1.0),
    (0.03, 200, 15, 1.0),
    (0.03, 300, 7, 5.0),
    (0.02, 400, 7, 5.0),
]
for learning_rate, trees, leaves, l2 in settings:
    model = train.make_boosting(trees, learning_rate)
    model[-1].set_params(max_leaf_nodes=leaves, l2_regularization=l2)
    print(learning_rate, trees, leaves, l2, validation_scores(model, train_df, val_df, columns))

0.05 100 15 1.0 {'roc_auc': 0.68, 'pr_auc': 0.237}
0.03 200 15 1.0 {'roc_auc': 0.68, 'pr_auc': 0.236}
0.03 300 7 5.0 {'roc_auc': 0.677, 'pr_auc': 0.23}
0.02 400 7 5.0 {'roc_auc': 0.677, 'pr_auc': 0.229}


No setting beats the current model.

## Idea 3: stacking

Four different models, plus a logistic regression that learns how to combine their
predictions. To train the combiner, each model predicts on data it was not trained on
(cross-validation). The folds are split **by patient**, for the same reason as the
train/test split.

This cell takes about 6 minutes on a laptop CPU.

In [7]:
from sklearn.ensemble import (
    ExtraTreesClassifier,
    RandomForestClassifier,
    StackingClassifier,
)
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import GroupKFold
from sklearn.pipeline import make_pipeline

make_columns = preprocess.make_column_transformer
estimators = [
    ("logistic", baseline.make_model()),
    ("boosting", train.make_boosting(50)),
    ("forest", make_pipeline(make_columns(), RandomForestClassifier(
        200, min_samples_leaf=50, n_jobs=-1, random_state=0))),
    ("extra_trees", make_pipeline(make_columns(), ExtraTreesClassifier(
        200, min_samples_leaf=50, n_jobs=-1, random_state=0))),
]
folds = list(GroupKFold(n_splits=4).split(
    train_df[columns], train_df[config.TARGET], groups=train_df["patient_nbr"]))

stack = StackingClassifier(estimators, final_estimator=LogisticRegression(max_iter=1000), cv=folds)
print("stacking:", validation_scores(stack, train_df, val_df, columns))
weights = stack.final_estimator_.coef_[0]
print("weights given to each model:",
      {name: round(float(w), 2) for (name, _), w in zip(estimators, weights)})

stacking: {'roc_auc': 0.683, 'pr_auc': 0.241}
weights given to each model: {'logistic': 0.27, 'boosting': 4.2, 'forest': 4.17, 'extra_trees': 3.29}


## Conclusion

| Idea | ROC-AUC | PR-AUC | Training time |
|---|---|---|---|
| Current model | 0.680 | 0.239 | ~3 s |
| + patient history | 0.684 | 0.239 | ~3 s |
| Tuned boosting (best) | 0.680 | 0.237 | ~10 s |
| Stacking, 4 models | 0.683 | 0.241 | ~6 min |

No idea improves the model by more than the ±0.01 that the score moves by chance.
All models learn from the same columns and reach the same limit: the information that
would predict readmission better (support at home, following the treatment) is not in
the data. The current model is kept: it is as good, much faster, and easier to explain.